# Email Spam Detection
Predict whether an email is spam from numeric features of the email.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split

from spam_utils import FEATURES, TARGET, make_pipeline

## 1. Load data

In [ ]:
df = pd.read_csv("data/email_spam_detection.csv")
print(df.shape)
df.head()

## 2. Data quality

In [ ]:
print(df.isna().sum())
print("Negative values:\n", (df.select_dtypes("number") < 0).sum())
print("Duplicate rows:", df.duplicated().sum())
df.describe().T

Missing `Email_Length` values and negative `Num_Special_Chars` values are handled inside the pipeline (negatives -> missing -> median).

## 3. Target balance

In [ ]:
print(df[TARGET].value_counts(normalize=True))
sns.countplot(x=TARGET, data=df)
plt.show()

## 4. Which columns carry signal?
Subject and sender domain look flat, the numeric features do not.

In [ ]:
df["Sender_Domain"] = df["Sender_Email"].str.extract(r"@(.+)$")[0]
print(df.groupby("Subject")[TARGET].mean().round(3))
print(df.groupby("Sender_Domain")[TARGET].mean().round(3))
df.groupby(TARGET)[FEATURES].mean().round(2)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, col in zip(axes, ["Email_Length", "Num_Links", "Num_Special_Chars", "Capital_Words"]):
    sns.boxplot(x=TARGET, y=col, data=df, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
sns.heatmap(df[FEATURES + [TARGET]].corr(), annot=True, fmt=".2f", cmap="Blues")
plt.show()

## 5. Train / test split

In [ ]:
X, y = df[FEATURES], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

## 6. Compare models (5-fold CV, F1)

In [ ]:
candidates = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, min_samples_leaf=3,
                                            class_weight="balanced", random_state=42, n_jobs=-1),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {n: cross_val_score(make_pipeline(m), X_train, y_train, cv=cv, scoring="f1").mean()
           for n, m in candidates.items()}
pd.Series(results).sort_values(ascending=False)

## 7. Evaluate best model on the hold-out set

In [ ]:
best_name = max(results, key=results.get)
best = make_pipeline(candidates[best_name]).fit(X_train, y_train)
pred = best.predict(X_test)
print(best_name)
print(classification_report(y_test, pred, target_names=["Not spam", "Spam"], digits=4))
print("ROC AUC:", roc_auc_score(y_test, best.predict_proba(X_test)[:, 1]))
ConfusionMatrixDisplay.from_predictions(y_test, pred)
plt.show()

The near-perfect scores are because this dataset looks synthetic and is almost perfectly separable. Real email would need text features.

## 8. Feature importance

In [ ]:
rf = best.named_steps["model"]
if hasattr(rf, "feature_importances_"):
    pd.Series(rf.feature_importances_, index=FEATURES).sort_values().plot.barh()
    plt.show()

## 9. Save the final model (trained on all data)

In [ ]:
final = make_pipeline(candidates[best_name]).fit(X, y)
joblib.dump(final, "spam_model.joblib")

## 10. Try a prediction

In [ ]:
sample = pd.DataFrame({"Email_Length": [190], "Num_Links": [6], "Num_Special_Chars": [15],
                       "Capital_Words": [30], "Has_Attachment": [1]})
model = joblib.load("spam_model.joblib")
print("Spam" if model.predict(sample)[0] == 1 else "Not spam", model.predict_proba(sample)[0][1])